# Lesson 01: Python & FastAPI Basics

## What You're Building

In this lesson you'll build a minimal FastAPI web service from scratch — the same foundation used by both the Ingestion API and Chat API in our Document Q&A app. By the end, you'll have a running service with a health endpoint, request validation via Pydantic, and environment-driven configuration.

FastAPI is the framework we chose for both backend services. It's a modern Python web framework that gives you automatic request validation, API documentation, and async support out of the box. If you've built HTTP services in Go with chi/gin or in TypeScript with Express, the concepts are the same — the syntax is just different.

## How This Fits in the App

Every endpoint in both services (POST /ingest, GET /documents, POST /chat, GET /health) is a FastAPI route. The patterns you learn here — route decorators, Pydantic models, async handlers, and settings — appear in every file you'll write in lessons 02-07.

In [ ]:
# Prerequisites — nothing external needed for this lesson
# Just make sure you have the packages installed:
# pip install fastapi uvicorn pydantic-settings
import fastapi
import pydantic_settings
print(f"FastAPI version: {fastapi.__version__}")
print("Ready to go!")

## Package Introductions

### FastAPI
FastAPI is a Python web framework built on top of Starlette (for HTTP) and Pydantic (for data validation). It was chosen for this project because:
- **Automatic validation** — request bodies are validated against Pydantic models before your code runs. In Go, you'd need to manually unmarshal and validate. In Express, you'd add middleware like `joi` or `zod`.
- **Async native** — route handlers can be `async def`, which matters when calling external services like Ollama and Qdrant.
- **Auto-generated docs** — visit `/docs` on any FastAPI app to get interactive Swagger UI. Free.

Key APIs you'll use:
- `FastAPI()` — create an app instance
- `@app.get("/path")`, `@app.post("/path")` — route decorators
- `HTTPException` — raise HTTP errors with status codes

### Pydantic
Pydantic is a data validation library. You define a class with type annotations and Pydantic ensures incoming data matches. Think of it as Go structs with `json` tags and built-in validation, or TypeScript interfaces that are enforced at runtime.

Key APIs:
- `BaseModel` — base class for request/response models
- `BaseSettings` (from pydantic-settings) — reads config from environment variables automatically

### Uvicorn
Uvicorn is an ASGI server — it's what actually runs your FastAPI app and handles HTTP connections. Think of it as the equivalent of Go's `http.ListenAndServe()` or Node's `app.listen()`. You won't interact with its API directly; you just point it at your app.

## Go/TS Comparison

| Concept | Go | TypeScript/Express | Python/FastAPI |
|---------|----|--------------------|----------------|
| Define a route | `r.Get("/health", handler)` | `app.get('/health', handler)` | `@app.get("/health")` decorator |
| Request body | `json.Unmarshal` into struct | `req.body` (unvalidated) or zod | Pydantic model as param (auto-validated) |
| Response | `json.NewEncoder(w).Encode(data)` | `res.json(data)` | `return data` (auto-serialized) |
| Config from env | `envconfig.Process(&cfg)` | `process.env.VAR` | `BaseSettings` class (auto-reads env) |
| Async I/O | goroutines (implicit) | `async/await` or callbacks | `async def` + `await` |

The biggest shift: In Go, concurrency is built into the runtime (goroutines). In Python, you explicitly mark functions as `async` and use `await` for I/O operations. If you forget `async`, the function blocks the entire server while waiting for a response.

## Build It

### Step 1: Create a minimal FastAPI app

This is the equivalent of `http.NewServeMux()` in Go or `express()` in Node.

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Ingestion API")

@app.get("/health")
async def health():
    return {"status": "ok"}

# In a real server, you'd run: uvicorn main:app --port 8000
# In a notebook, we can test using FastAPI's test client:
from fastapi.testclient import TestClient
client = TestClient(app)

response = client.get("/health")
print(f"Status: {response.status_code}")
print(f"Body: {response.json()}")

### Step 2: Add a Pydantic request model

Pydantic models define the shape of your request data. When a request comes in, FastAPI automatically:
1. Parses the JSON body
2. Validates it against your model
3. Returns a 422 error if validation fails

This is like defining a Go struct with `json` tags — but validation happens automatically. No `if err := json.Unmarshal(...); err != nil` needed.

In [ ]:
from pydantic import BaseModel

class ChatRequest(BaseModel):
    question: str
    collection: str | None = None  # Optional with default None

# Test it — valid input
valid = ChatRequest(question="What is revenue?")
print(f"Question: {valid.question}")
print(f"Collection: {valid.collection}")  # None (default)

# Test it — with collection
with_collection = ChatRequest(question="What is revenue?", collection="documents")
print(f"Collection: {with_collection.collection}")

# Test it — invalid input (missing required field)
try:
    bad = ChatRequest()
except Exception as e:
    print(f"Validation error: {e}")

### Step 3: Use the model in a route

When you put a Pydantic model as a function parameter, FastAPI knows it's the request body. It parses, validates, and passes it to your function — or returns 422 if invalid.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.testclient import TestClient

app = FastAPI(title="Chat API")

class ChatRequest(BaseModel):
    question: str
    collection: str | None = None

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/chat")
async def chat(request: ChatRequest):
    return {
        "echo": request.question,
        "collection": request.collection or "default",
    }

client = TestClient(app)

# Valid request
response = client.post("/chat", json={"question": "What is revenue?"})
print(f"Status: {response.status_code}")
print(f"Body: {response.json()}")

# Invalid request — missing question
response = client.post("/chat", json={})
print(f"\nInvalid request status: {response.status_code}")
print(f"Error detail: {response.json()['detail'][0]['msg']}")

### Step 4: Environment-driven configuration with pydantic-settings

In Go you might use `envconfig` or `viper` to read environment variables into a config struct. Python's `pydantic-settings` does the same thing — you define a class with defaults, and it automatically reads matching environment variables.

The field name `qdrant_host` matches the env var `QDRANT_HOST` (case-insensitive). This is how both services get their config.

In [ ]:
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    ollama_base_url: str = "http://host.docker.internal:11434"
    embedding_model: str = "nomic-embed-text"
    qdrant_host: str = "qdrant"
    qdrant_port: int = 6333
    collection_name: str = "documents"
    chunk_size: int = 1000
    chunk_overlap: int = 200
    max_file_size_mb: int = 50

settings = Settings()

print(f"Ollama URL: {settings.ollama_base_url}")
print(f"Qdrant: {settings.qdrant_host}:{settings.qdrant_port}")
print(f"Chunk size: {settings.chunk_size}")
print(f"Collection: {settings.collection_name}")

# If you set CHUNK_SIZE=500 in your environment, it would override the default.
# Try it: import os; os.environ["CHUNK_SIZE"] = "500"
# Then re-create Settings() and check settings.chunk_size

> **Pitfall:** Claude Code often generates synchronous route handlers (`def health()` instead of `async def health()`). For simple endpoints that don't do I/O, it doesn't matter. But for handlers that call Ollama or Qdrant (which we'll build later), forgetting `async` will block the entire server during the request. Always use `async def` for FastAPI routes.

## Experiment

Try these modifications to build intuition:

In [ ]:
# Experiment 1: Add a query parameter
# In Go: r.URL.Query().Get("name")
# In Express: req.query.name
# In FastAPI: just add it as a function parameter

@app.get("/greet")
async def greet(name: str = "world"):
    return {"message": f"Hello, {name}!"}

client = TestClient(app)
print(client.get("/greet").json())
print(client.get("/greet?name=Kyle").json())

In [ ]:
# Experiment 2: Add validation constraints
# Pydantic lets you add constraints that Go would need custom validation for

from pydantic import BaseModel, Field

class IngestRequest(BaseModel):
    filename: str = Field(min_length=1, max_length=255)
    max_pages: int = Field(default=100, ge=1, le=1000)

# Valid
print(IngestRequest(filename="report.pdf").model_dump())

# Invalid — empty filename
try:
    IngestRequest(filename="")
except Exception as e:
    print(f"Validation error: {e}")

## Check Your Understanding

1. **In your own words, what does a Pydantic model do that a plain Python dictionary doesn't?**

2. **Why do we use `async def` for route handlers instead of regular `def`? When does it matter?**

3. **How does `pydantic-settings` know to read `QDRANT_HOST` from the environment for the `qdrant_host` field?**